In [2]:
!pip install akshare --upgrade

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 82.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 65.7 MB/s eta 0:00:00
  Created wheel for jsonpath: filename=jsonpath-0.82.2-py3-none-any.whl size=5615 sha256=6c55168385ed0ed4b4b1509b9b7204696f65b70b498848f8a9aef811c46a12c2
  Stored in directory: /root/.cache/pip/wheels/73/76/e2/980a29341fe37a583ada29594ed529708d5e8e2c0f9d97c3cc
Successfully built jsonpath


In [3]:
import akshare as ak
import numpy as np
import pandas as pd

In [4]:
def crr_vanilla_call(S, K, sigma, r, T, n, b=0.0, verbose=False):
  '''
  args:
  S: 当前股票价格，K=执行价格，sigma=年化波动率，r=年化利率，T=总时间（年），n=步数，b=年化股息率
  return:
  当前价值，float
  '''
  dt = T / n
  u = np.exp(sigma * np.sqrt(dt))
  d = 1 / u
  p = (np.exp( (r - b) * dt ) - d) / (u - d)

  i = np.arange(n+1)
  S_terminal = S * (u**i) * (d ** (n-i)) # 终点股票价格数组
  V = np.maximum(S_terminal-K, 0) # 终期价值数组,到期时二选一,取大的

  # 把相邻元素配对计算，使 V 的长度减少1，并打印长度观察变化。
  for j in range(n-1, -1, -1):
    V = np.exp(- r * dt) *(p * V[1:] + (1-p) * V[:-1]) # V = 折现因子 * 加权数组，每轮长度减少1的当前价值数组
    if verbose:
      print(f"当前层:{j}, V的长度: {len(V)}")

  return float(V[0])
# 调试调用
print(crr_vanilla_call(S=100,K=100,sigma=0.2,r=0.05,T=1,n=3, verbose=True))

n_values = [10, 50, 200, 1000]
# 其他参数
params = {
    "S" : 100,
    "K" : 100,
    "sigma" : 0.2,
    "r" : 0.05,
    "T": 1}

'''
price = np.zeros(len(n_values))
# 比较不同步数的计算结果
for k, current_n in enumerate(n_values):
  price[k] = crr_vanilla_call(**params, n=current_n)
  print(f"当前步数:{current_n}, 价格：{price[k]:.6f}")

changes = np.abs(price[1:] - price[:-1])
for k, change in enumerate(changes):
  print(f"n{n_values[k]} -> {n_values[k+1]}, 变化量{change:.6f}")
'''

def compare_crr_steps(n_values, params):
  """
  args:
  n_values=不同步数，params = 其他不变的参数
  return：
  每个步数对应的计算结果，相邻结果的绝对变化量
  """
  prices = np.zeros(len(n_values))
  for k, current_n in enumerate(n_values):
    prices[k] = crr_vanilla_call(**params, n=current_n)

  changes = np.abs(prices[1:] - prices[:-1])
  return prices, changes

prices, changes = compare_crr_steps(n_values, params)
print(prices, changes)




当前层:2, V的长度: 3
当前层:1, V的长度: 2
当前层:0, V的长度: 1
11.043871091951113
[10.25340904 10.41069154 10.44059126 10.4485841 ] [0.1572825  0.02989972 0.00799284]


In [5]:
# 其他参数
params_con = {
    "S" : 100,
    "K" : 100,
    "sigma" : 0.2,
    "r" : 0.05,
    "T" : 1,
    "b" : 0.0,
    "P_R" : 108.0,
    "coupon" : 0.01,
    "spread" : 0.02, # 额外折现率 信用利差？
    "lo" : 70.0, # 插值下界
    "hi" : 130.0, # 插值上界
    "t_conv": 0.5, # 开始允许转换的时间
    "t_put": 0.5 # 允许回售的时间
    }

def parity(S, face, K):
  return S * face / K

def crr_convertible_basic(S, K, sigma, r, T,
                          b, P_R, coupon, spread, lo, hi, t_conv,t_put,
                          n, C1=130, C2=70, face=100, P_put=103.0, verbose=False):
  '''
  args:
  S: 当前股票价格，K=执行价格，sigma=年化波动率，r=年化利率，T=总时间（年），n=步数，b=年化股息率
  face=转债面值, P_R=到期赎回金额, coupon = 年化票面利率, spread = 额外折现率 信用利差？,
  lo = 插值下界, hi = 插值上界, C1 = 强赎触发线（平价），C2 = 回售触发线（平价），
  P_put=回售价格
  return:
  当前价值，float
  '''
  dt = T / n
  u = np.exp(sigma * np.sqrt(dt))
  d = 1 / u
  p = (np.exp( (r - b) * dt ) - d) / (u - d)
  interest_per_step = coupon * face * dt

  i_current = np.arange(n+1)
  S_terminal = S * (u**i_current) * (d ** (n-i_current)) # 终点股票价格数组
  parity_terminal =  parity(S_terminal, face, K)
  V = np.maximum(parity_terminal, P_R) # 终期价值数组,到期时二选一,取大的

  for j in range(n-1, -1, -1): # 从终期价值回溯
    i_current = np.arange(j+1)
    S_current = S * (u ** i_current) * (d ** (j-i_current))
    parity_current = parity(S_current, face, K)
    t_current = j * dt
    # 折现率
    P_ct = np.clip(((parity_current - lo) / (hi - lo)), 0, 1)
    Df = r * P_ct + (1-P_ct) * (r + spread)
    V_EU = np.exp(- Df * dt) *(p * V[1:] + (1-p) * V[:-1]) + interest_per_step

    if t_current >= t_conv: # 时间到了才能换
      hit_call = parity_current > C1 # 检测强赎条款
      V = np.maximum(V_EU, parity_current)
      V[hit_call] = parity_current[hit_call]
    else:
      V = V_EU

    if t_current >= t_put:
      put = parity_current < C2 # 检测回售条款
      V[put] = np.maximum(V_EU[put], P_put)

    if verbose:
      print(f"当前层:{j}, V的长度: {len(V)}")

  return float(V[0])


print(crr_convertible_basic(**params_con, n=5, verbose = True))

当前层:4, V的长度: 5
当前层:3, V的长度: 4
当前层:2, V的长度: 3
当前层:1, V的长度: 2
当前层:0, V的长度: 1
109.17928591664443


In [6]:
bond_zh_cov_df = ak.bond_zh_cov()
print(bond_zh_cov_df)

 67%|██████▋   | 2/3 [00:07<00:03,  3.58s/it]/usr/local/lib/python3.12/dist-packages/akshare/bond/bond_zh_cov.py:342: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  big_df = pd.concat(objs=[big_df, temp_df], ignore_index=True)
                                             

        债券代码   债券简称        申购日期    申购代码  申购上限    正股代码  正股简称    正股价     转股价  \
0     123276  久吾转02  2026-07-20  370631   100  300631  久吾高科  21.89   23.78   
1     118073   赛斯转债  2026-07-17  718480   100  688480   赛恩斯  67.84   88.16   
2     113708  曙26转债  2026-07-15  754019   100  603019  中科曙光  87.89  108.89   
3     110102   江农转债  2026-07-14  733389   100  600389  江山股份  16.87   20.64   
4     123275   肇民转债  2026-07-09  371000   100  301000  肇民科技  25.78   33.80   
...      ...    ...         ...     ...   ...     ...   ...    ...     ...   
1030  110227   赤化转债  2007-10-10  733227   100  600227   赤天化   3.06     NaN   
1031  126006  07深高债  2007-10-09  733548   100  600548   深高速   8.46     NaN   
1032  110971   恒源转债  2007-09-24  733971   100  600971  恒源煤电   7.64     NaN   
1033  110567   山鹰转债  2007-09-05  733567   100  600567  山鹰国际   1.33     NaN   
1034  110026   中海转债  2007-07-02  733026   100  600026  中远海能  14.21     NaN   

         转股价值    债现价  转股溢价率 原股东配售-股权登记日  原股东配售-每股配售额    发行规模   

In [7]:
# 筛出的目标转债行
row = bond_zh_cov_df[bond_zh_cov_df["债券简称"] == "健帆转债"]
print(row)

K_bond = row["转股价"].values[0] # 转股价
stock_code = row["正股代码"].values[0]
print(stock_code)



       债券代码  债券简称        申购日期    申购代码  申购上限    正股代码  正股简称   正股价    转股价  \
455  123117  健帆转债  2021-06-23  370529   100  300529  健帆生物  17.4  38.15   

        转股价值      债现价   转股溢价率 原股东配售-股权登记日  原股东配售-每股配售额  发行规模      中签号发布日  \
455  45.6094  115.693  153.66  2021-06-22       1.2421  10.0  2021-06-25   

          中签率        上市时间 信用评级  
455  0.001814  2021-07-12   AA  
300529


In [8]:
stock_df=ak.stock_zh_a_daily(symbol="sz300529", start_date="20250716", end_date="20260716", adjust="qfq")
print(stock_df)

           date   open   high    low  close      volume       amount  \
0    2025-07-16  21.74  21.85  21.62  21.78   4842270.0  108227878.0   
1    2025-07-17  21.79  22.03  21.73  21.99   5544738.0  124782210.0   
2    2025-07-18  21.98  22.18  21.90  22.17   6328358.0  143461705.0   
3    2025-07-21  22.18  22.25  22.04  22.09   5861685.0  133393488.0   
4    2025-07-22  22.07  22.30  21.94  22.15   7084612.0  160842891.0   
..          ...    ...    ...    ...    ...         ...          ...   
238  2026-07-10  15.84  16.29  15.70  16.05   6902530.0  110732784.0   
239  2026-07-13  15.98  16.38  15.89  16.23   8047685.0  130429425.0   
240  2026-07-14  16.29  17.14  16.15  16.55   9946061.0  165989038.0   
241  2026-07-15  16.53  17.15  16.38  17.07  11259591.0  190854102.0   
242  2026-07-16  16.90  17.83  16.81  17.70  12362896.0  215534192.0   

     outstanding_share  turnover  
0          511585981.0  0.009465  
1          511585981.0  0.010838  
2          511585981.0  0.0123

In [9]:
bond_profile = ak.bond_cb_profile_sina(symbol="sz123117")
maturity = bond_profile[bond_profile["item"] == "到期日"]["value"].values[0]

In [10]:
# bond_cov_comparison_df = ak.bond_cov_comparison() # 东方财富网接口特有的连不上，手抄数据了

In [11]:
row_comparision = bond_zh_cov_df[bond_zh_cov_df["债券简称"] == "健帆转债"]
P_R_bond = 110 #row_comparision["到期赎回价"].values[0]

110 49.6 26.71


In [12]:
# 算一下到期年限
Pt = stock_df["close"].to_numpy()
r_t = np.log(Pt[1:] / Pt[:-1])
sigma_year = np.std(r_t, ddof=1) * np.sqrt(252)
print(sigma_year)

time_delta = pd.to_datetime(maturity) - pd.Timestamp.today()
T_bond = (time_delta.days)/365
print(T_bond)

# 把 S 弄出来
S_bond = row["正股价"].values[0]

# maturity = bond_profile[bond_profile["item"] == "到期日"]["value"].values[0]

# 全部塞进去
params_cb = {
    "S" : S_bond,
    "K" : K_bond,
    "sigma" : sigma_year,
    "r" : 0.02, # 估算常数
    "T" : T_bond,
    "b" : 0.0,
    "P_R" : P_R_bond,
    "coupon" : 0.018, # 手抄
    "spread" : 0.01, # 额外折现率 信用利差？
    "lo" : 70.0, # 插值下界
    "hi" : 130.0, # 插值上界
    "t_conv": 0.0, # 开始允许转换的时间
    "t_put": 0.0 # 允许回售的时间
    }

print(crr_convertible_basic(**params_cb, n=5, verbose = True))

0.2511534722154877
0.9232876712328767
当前层:4, V的长度: 5
当前层:3, V的长度: 4
当前层:2, V的长度: 3
当前层:1, V的长度: 2
当前层:0, V的长度: 1
108.63866152986255


### 为什么少8块？
- 下修条款没做
- 模型精度？可以玩一下TF
- 其他误差？

但这也差太多了

### 关键结果

- 模型理论价 $\approx 108$ 元，市场债现价 $115.7$ 元，**低约 8 元**。
- 这只券深度价外，理论价几乎等于纯债底（到期赎回价折现 + 票息）。市场价高出的部分主要来自**下修条款预期**：市场押注发行人下调转股价 $K$，让转股期权重新有价值。暂时没做下修，系统性低估。
- 敏感度分析佐证：调参后最多到 $109$，补不平这 $8$ 元，说明缺的是结构性期权而非参数没调好。
- 附带观察：对这只深度价外券，树的层数 $n$ 从 $5$ 到 $1000$ 价格几乎不变，因为它本质是一张债、期权成分极小；平价 $100$ 附近的券才需要大 $n$

### 下一步
1. 加下修条款

2. 整理成可以复用的函数，拉一张平价接近100的券，重新跑一遍

3. 实现 TF 模型：价值拆成 $B+E$ 两条数组分别倒推，比"平价插值折现"更规范地处理信用风险。

In [13]:
pd.set_option("display.max_colwidth",None)
print(bond_profile)

        item                                                         value
0       债券名称                         2021年健帆生物科技集团股份有限公司创业板向不特定对象发行可转换公司债券
1       债券简称                                                          健帆转债
2       债券代码                                                      sz123117
3       债券类型                                                        可转换企业债
4    债券面值（元）                                                           100
5    债券年限（年）                                                             6
6    票面利率（%）                                                            --
7        到期日                                                    2027-06-23
8        兑付日                                                    2027-06-23
9        摘牌日                                                            --
10      计息方式                                                          递进利率
11      利率说明  第一年为0.30%，第二年为0.50%，第三年为1.00%，第四年为1.50%，第五年为1.80%，第六年为2.00%。
12      付息方式             

In [14]:
print(params_cb)

{'S': np.float64(17.4), 'K': np.float64(38.15), 'sigma': np.float64(0.2511534722154877), 'r': 0.02, 'T': 0.9232876712328767, 'b': 0.0, 'P_R': 110, 'coupon': 0.018, 'spread': 0.01, 'lo': 70.0, 'hi': 130.0, 't_conv': 0.0, 't_put': 0.0}
